# NovaClaw Mobile-Agentic Fine-Tuning (Gemma 4 E2B / E4B)

Runs the full 3-stage pipeline on a Colab/Linux GPU (T4 16GB, L4 24GB, or A100):
1. Install dependencies & build the NovaClaw error-recovery + Android-grounding dataset
2. Baseline evaluation on `google/gemma-4-E2B-it`
3. Stage 1: QLoRA SFT (`train_sft.py`)
4. Stage 2: GRPO Verifiable Rewards (`train_grpo.py`)
5. Post-training held-out evaluation (`eval_agent.py`)
6. Stage 3: Export to `.litertlm` (`export_litertlm.sh`) for NovaClaw Android

In [ ]:
!nvidia-smi
!pip install -q -U torch torchvision transformers>=5.10.1 trl>=0.16.0 peft>=0.19.0 accelerate datasets bitsandbytes huggingface_hub uv

In [ ]:
from huggingface_hub import login
# Accept the Gemma 4 license on Hugging Face before logging in:
# https://huggingface.co/google/gemma-4-E2B-it
login()

In [ ]:
# Clone your NovaClaw repo (or upload the training/ folder directly to Colab)
!git clone -b gemma4-agent-training https://github.com/Maserbytes/NovaClaw.git || true
%cd /content/NovaClaw/training

In [ ]:
!python build_dataset.py --out-dir data --sft-count 1200 --grpo-count 400 --eval-count 120
!python -m unittest test_training_suite.py -v

In [ ]:
MODEL_ID = 'google/gemma-4-E2B-it'  # Or 'google/gemma-4-E4B-it' on >=24GB GPU
!python eval_agent.py --model {MODEL_ID} --load-4bit --out-report outputs/eval_before.json

In [ ]:
!python train_sft.py --model {MODEL_ID} --epochs 2 --batch-size 2 --grad-accum 8 --lora-r 16 --lora-alpha 32

In [ ]:
!python train_grpo.py --model outputs/merged_sft --max-steps 80 --num-generations 4 --batch-size 4

In [ ]:
!python eval_agent.py --model outputs/merged_final --load-4bit --enable-thinking --out-report outputs/eval_after.json

In [ ]:
!bash export_litertlm.sh outputs/merged_final outputs/litert_out E2B